First notebook cell: test Python

In [1]:
print("HR Employee 360 project started!")

HR Employee 360 project started!


Create Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HR Employee 360")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.8"
    )
    .getOrCreate()
)

print("Spark started successfully!")
print("Spark version:", spark.version)


Spark started successfully!
Spark version: 4.2.0


Check that Spark received the package

In [2]:
print(
    spark.sparkContext.getConf().get(
        "spark.jars.packages"
    )
)


org.postgresql:postgresql:42.7.8


Test PySpark

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HR Employee 360")
    .getOrCreate()
)

print("Spark started successfully!")


Spark started successfully!


In [4]:
spark.version

'4.2.0'

In [ ]:
Load your PostgreSQL configuration - config/database_config.py

In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.database_config import get_database_config

config = get_database_config()

print("PostgreSQL configuration loaded")
print("Host:", config["host"])
print("Port:", config["port"])
print("Database:", config["database"])
print("User:", config["user"])
print("JDBC URL:", config["jdbc_url"])


PostgreSQL configuration loaded
Host: localhost
Port: 5432
Database: HR
User: vikas
JDBC URL: jdbc:postgresql://localhost:5432/HR


Test PostgreSQL through PySpark

In [7]:
employees_df = (
    spark.read
    .format("jdbc")
    .option("url", config["jdbc_url"])
    .option("dbtable", "hr_data.employees")
    .option(
        "user",
        config["connection_properties"]["user"]
    )
    .option(
        "password",
        config["connection_properties"]["password"]
    )
    .option(
        "driver",
        config["connection_properties"]["driver"]
    )
    .load()
)


In [8]:
employees_df.show()

+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|employee_id|first_name|last_name|gender|       dob|department_id|job_id|manager_id| hire_date|employment_type|status|
+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|       E001|     Aarav|   Sharma|  Male|1995-03-15|          D01|   J01|      E010|2020-05-12|      Full-Time|Active|
|       E002|     Priya|    Patel|Female|1992-08-21|          D02|   J02|      E011|2019-02-15|      Full-Time|Active|
|       E003|     Rahul|    Verma|  Male|1988-01-10|          D03|   J03|      E012|2016-06-20|      Full-Time|Active|
|       E004|      Neha|    Singh|Female|1997-02-14|          D04|   J04|      E013|2022-07-18|      Full-Time|Active|
|       E005|     Rohan|    Mehta|  Male|1994-08-01|          D05|   J05|      E014|2019-09-01|      Full-Time|Active|
|       E006|    Ananya|    Gupta|Female|1996-05

Check all PostgreSQL tables
Before building Employee 360, let's find out what tables you actually have.

You can use PostgreSQL through JDBC with a query.

In [9]:
tables_df = (
    spark.read
    .format("jdbc")
    .option("url", config["jdbc_url"])
    .option(
        "dbtable",
        "(SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'public') AS tables"
    )
    .option(
        "user",
        config["connection_properties"]["user"]
    )
    .option(
        "password",
        config["connection_properties"]["password"]
    )
    .option(
        "driver",
        config["connection_properties"]["driver"]
    )
    .load()
)

tables_df.show(truncate=False)


+----------+
|table_name|
+----------+
|employees |
+----------+



Step 9 — Don't build the Employee 360 yet
This is important.

Before we write all the joins and business metrics, we need to know the actual table and column names in your PostgreSQL database.

For example, your tables might be:

employees
departments
salary
performance
promotions

But your actual project could use completely different names.

We need to inspect them first.

For each table, run:

In [10]:
employees_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- department_id: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- status: string (nullable = true)



In [11]:
employees_df.show(5)

+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|employee_id|first_name|last_name|gender|       dob|department_id|job_id|manager_id| hire_date|employment_type|status|
+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|       E001|     Aarav|   Sharma|  Male|1995-03-15|          D01|   J01|      E010|2020-05-12|      Full-Time|Active|
|       E002|     Priya|    Patel|Female|1992-08-21|          D02|   J02|      E011|2019-02-15|      Full-Time|Active|
|       E003|     Rahul|    Verma|  Male|1988-01-10|          D03|   J03|      E012|2016-06-20|      Full-Time|Active|
|       E004|      Neha|    Singh|Female|1997-02-14|          D04|   J04|      E013|2022-07-18|      Full-Time|Active|
|       E005|     Rohan|    Mehta|  Male|1994-08-01|          D05|   J05|      E014|2019-09-01|      Full-Time|Active|
+-----------+----------+---------+------+-------